<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# Lab 7 — Part 2: TF-IDF search with DuckDB

This notebook implements a lexical retrieval pipeline. Scikit-learn transforms passages and questions into TF-IDF vectors; DuckDB stores sparse weights and ranks documents by the dot product between query and document vectors. Unlike dense semantic embeddings, TF-IDF primarily matches shared terms.


## Software requirements and environment setup

This project requires Python 3.11 or newer. The `pyproject.toml` file declares the dependencies used here, including scikit-learn, SciPy, pandas, Hugging Face Datasets, and DuckDB. From a terminal, first change to the project directory containing `pyproject.toml`.

Install `uv` if it is not already available:

- macOS / Linux (Terminal): `curl -LsSf https://astral.sh/uv/install.sh | sh`
- Windows (PowerShell): `powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"`

Then create or update the virtual environment and install the dependencies declared in `pyproject.toml`:

```bash
uv sync
uv run jupyter lab
```

Open this notebook in JupyterLab. If you launch Jupyter from another Python environment and need this project environment to appear as a selectable kernel, register it optionally with:

```bash
uv run python -m ipykernel install --user --name lab7 --display-name "Python (lab7)"
```

Select `Python (lab7)` in the notebook kernel menu. The Python package `duckdb` is already listed in the project dependencies and is installed by `uv sync`; the separate DuckDB command-line client is not required.


## 1. Import dependencies

The notebook uses scikit-learn for TF-IDF, Hugging Face Datasets for the example corpus, pandas for tabular data, and DuckDB for SQL-based storage and ranking. The project environment described in Part 1 includes these dependencies.


In [ ]:
# Import TF-IDF, sparse-matrix, data-loading, and DuckDB dependencies.
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from scipy.sparse._csr import csr_matrix
import duckdb
import pandas as pd
import numpy as np
import scipy

## 2. Load the corpus

The passage split supplies text documents. The test question-answer split is also loaded for later exploration; the current example uses a question written directly in the notebook.


In [ ]:
# Load the passages to index and the held-out QA split.
corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")["test"]

## 3. Build the TF-IDF store

`DBStore` fits a `TfidfVectorizer` on the passage corpus, saves the vocabulary and nonzero term weights in separate tables, and scores a query by summing the products of matching query/document weights. `min_df`, `max_df`, and `max_features` control vocabulary filtering.


In [ ]:
class DBStore:
    def __init__(self, db_path: str = "./database-p2.db", max_feature: int = 20000) -> None:
        """
        Initialize 

        Args:
            db_path: Path to database file
        """
        self.conn = duckdb.connect(db_path)
        self.max_feature = max_feature
        # Filter very rare/common terms and cap vocabulary size for efficiency.
        self.vectorizer = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            min_df=2,                             # ignore terms appearing in fewer than 2 passages
            max_df=0.5,                           # ignore terms appearing in more than 50% of passages
            max_features=self.max_feature,        # keep only the top N terms by corpus frequency
        )
        self._create_tables()

    def _create_tables(self) -> None:
        """Create the table if it doesn't exist"""
        try:
            # Use IF NOT EXISTS to avoid errors when table already exists
            create_table_query = f"""
            CREATE TABLE IF NOT EXISTS content (
                doc_id UInt32,
                content String
            );

            CREATE OR REPLACE TABLE vocab(
                term_id UInt32, 
                term String
            );

            CREATE OR REPLACE TABLE tfidf(
                doc_id UInt32, 
                term_id UInt32, 
                w FLOAT
            );
            """
            self.conn.execute(create_table_query)
        except Exception as e:
            print(f"Warning when creating table: {str(e)}")
            print("Continuing with existing table...")

    def _generate_tfidf(self, tfidf: csr_matrix) -> None:
        """Store the nonzero TF-IDF weights in sparse row/term form."""
        try:
            # Store only nonzero entries to keep the database representation sparse.
            coo = tfidf.tocoo()
            tf_df = pd.DataFrame({"doc_id": coo.row, "term_id": coo.col, "w": coo.data.astype("float32")})
            self.conn.execute("""
            DELETE FROM tfidf;
            INSERT INTO tfidf SELECT * FROM tf_df;
            """)
        except Exception as e:
            print(f"Warning when generating tf-idf: {str(e)}")
            
    def _generate_vocab(self, corpus: pd.DataFrame) -> None:
        """Fit the vectorizer, persist its vocabulary, and store document weights."""
        docs = list(corpus["passage"])
        try:
            # Fit corpus vocabulary and create one sparse TF-IDF row per passage.
            X = self.vectorizer.fit_transform(docs)
            vocab_df = pd.DataFrame({
                "term_id": range(len(self.vectorizer.get_feature_names_out())),
                "term": self.vectorizer.get_feature_names_out(),
            })
            self.conn.execute("""
            DELETE FROM vocab;
            INSERT INTO vocab SELECT * FROM vocab_df
            """)
            self._generate_tfidf(X)
        except Exception as e:
            print(f"Warning when generating vocab: {str(e)}")

    def load_dataset(self, dataset: Dataset) -> None:
        try:
            # Convert the dataset so DuckDB can read it as a registered DataFrame.
            df_passage = dataset.to_pandas()
            # Load in database
            query ="""
            DELETE FROM content;
            INSERT INTO content (doc_id, content)
            SELECT 
                id AS doc_id,
                passage AS content
            FROM df_passage
            """
            self.conn.execute(query)
            self._generate_vocab(df_passage)
        except Exception as e:
            print(f"Warning when loading dataset: {str(e)}")

    def search(self, question: str, topk: int = 5) -> pd.DataFrame: 
        """Rank stored passages by TF-IDF dot product with the query."""
        q = self.vectorizer.transform([question]).tocoo()
        # Convert query nonzeros into a table DuckDB can join by term ID.
        qdf = pd.DataFrame({"term_id": q.col, "qw": q.data})
        return self.conn.execute(f"""
        SELECT 
            p.content, 
            SUM(t.w * qdf.qw) AS score,
            RANK() OVER (ORDER BY score DESC) AS rank
        FROM tfidf t
        JOIN qdf USING (term_id)
        JOIN content p USING (doc_id)
        GROUP BY p.doc_id, p.content
        ORDER BY score DESC
        LIMIT ?
        """, [topk]).df()

    def close(self) -> None:
        self.conn.close()

## 4. Run a search

The query is transformed using the already-fitted vocabulary. Rows are ordered by descending dot-product score; larger scores indicate stronger lexical overlap under this TF-IDF representation.


In [ ]:
# Fit TF-IDF on the passage corpus, search the example question, then close the connection.
db = DBStore()
db.load_dataset(corpus)

question = 'Was Abraham Lincoln the sixteenth President of the United States?'
results = db.search(question)
db.close()

In [ ]:
results

## Student exercises

Run the notebook from the beginning before starting. Use `db` and `results` from the example. Remember that retrieval scores rank candidates; inspect the text to judge whether a passage actually answers a question.

### Exercise 1 — Compare query wording

Try at least three differently worded questions about the same topic. Compare top-ranked passages and scores. Which terms seem to influence the ranking?


### Exercise 2 — Change the retrieval depth

Search with `topk=1`, `topk=3`, and `topk=10`. Does a larger result set always provide more useful evidence? Explain.


### Exercise 3 — Inspect vocabulary filtering

Change `min_df`, `max_df`, or `max_features` in `DBStore`, reload the corpus, and compare results. Which terms are removed or retained? How does this affect the ranking?


### Exercise 4 — Evaluate precision at k

For one question, manually decide which retrieved passages are relevant. Calculate precision@k for a few values of k, then discuss what this measure misses (for example, relevant passages ranked below k).


In [ ]:
# Reopen the persisted DuckDB file; the fitted vectorizer remains on the `db` object.
db.conn = duckdb.connect("./database-p2.db")

# Try alternative formulations and inspect the top-ranked passages.
alternative_questions = [
    "Was Abraham Lincoln elected as the sixteenth US president?",
    "Which office did Abraham Lincoln hold?",
    "Tell me about the history of the United States.",
]
for alternative in alternative_questions:
    print(f"\nQuestion: {alternative}")
    alternative_results = db.search(alternative, topk=3)
    print(alternative_results[["score", "rank", "content"]].to_string(index=False, max_colwidth=180))

In [ ]:
# Compare how many results are returned for different top-k values.
for k in [1, 3, 10]:
    top_results = db.search(question, topk=k)
    print(f"\nTop {k} results (returned {len(top_results)}):")
    print(top_results[["score", "rank", "content"]].to_string(index=False, max_colwidth=140))


In [ ]:
# TODO: Mark the returned ranks that are relevant to your question.
relevant_ranks = {1, 2}  # Replace these with your own relevance judgments.
for k in [1, 3, 5]:
    relevant_in_top_k = sum(rank in relevant_ranks for rank in results["rank"].head(k))
    precision_at_k = relevant_in_top_k / k
    print(f"Precision@{k}: {precision_at_k:.3f}")

# Close the database connection after completing the exercises.
db.close()

## Exercise 5 — Explore the DuckDB database

Inspect the database created by `DBStore`. List its tables, examine their schemas, and count rows in each table. The sparse `tfidf` table stores one row per nonzero document-term weight; `vocab` maps numeric term IDs to words, and `content` stores passage IDs and text.

Then inspect a few vocabulary entries and find terms with the largest total TF-IDF weight across the corpus. Does a large total weight necessarily mean a term is useful for a particular query? Explain.


In [ ]:
# Connect directly to the DuckDB file produced by the main workflow.
explore_conn = duckdb.connect("./database-p2.db")

# List tables and inspect their columns and DuckDB types.
print("Tables:")
print(explore_conn.execute("SHOW TABLES").df())
for table_name in ["content", "vocab", "tfidf"]:
    print(f"\nSchema for {table_name}:")
    print(explore_conn.execute(f"DESCRIBE {table_name}").df())
    row_count = explore_conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"Rows: {row_count:,}")

# Inspect a few ID-to-term mappings.
print("\nVocabulary sample:")
print(explore_conn.execute("SELECT * FROM vocab ORDER BY term_id LIMIT 10").df())

# Aggregate weights to see which terms have the largest total corpus weight.
print("\nLargest aggregate TF-IDF weights:")
print(explore_conn.execute("""
    SELECT v.term, SUM(t.w) AS total_weight, COUNT(*) AS document_count
    FROM tfidf AS t
    JOIN vocab AS v USING (term_id)
    GROUP BY v.term
    ORDER BY total_weight DESC
    LIMIT 10
""").df())

explore_conn.close()
